# CSE 151B Competition — M4 Mac (MLX) Local Version

This is the **Apple Silicon** version of the starter notebook, adapted to run locally on an M-series Mac using the GPU via Apple's **MLX** framework.

### What changed vs the original

| Original (CUDA) | This version (M4) | Why |
|---|---|---|
| `vllm` | `mlx-lm` | vLLM is CUDA-only; MLX is Apple's native framework and uses the unified GPU |
| `bitsandbytes` INT8 | MLX 4-bit quant | bitsandbytes needs CUDA; MLX has its own quant format |
| `SamplingParams` (vLLM) | `mlx_lm.generate` kwargs | Different API, same concept |
| `CUDA_VISIBLE_DEVICES` | (removed) | MLX uses the Apple GPU automatically |

### ⚠️ Expect slower inference
A 4B thinking model on M-series silicon is significantly slower than on a CUDA GPU. Thinking models often emit **5k–20k+ reasoning tokens per question**, so a full dataset run could take hours. Use this for debugging and small-scale experiments locally; run the full sweep on a CUDA GPU.

### Dataset
Put `public.jsonl` (and `judger.py` for scoring) next to this notebook before running.

## 1. Environment Setup

Run the install cell **once**, then comment it out. MLX only works on Apple Silicon (M1/M2/M3/M4) — it will fail on Intel Macs or Linux.

> After the first install, restart the kernel so the new packages are picked up.

In [14]:
# ─── First-time install (comment out after running once) ───
import platform, sys
assert platform.system() == "Darwin" and platform.machine() == "arm64", \
    "This notebook requires Apple Silicon (M1/M2/M3/M4). For CUDA GPUs, use the original starter."

!.cse151b/bin/python -m pip install --quiet --upgrade pip
!.cse151b/bin/python -m pip install --quiet mlx mlx-lm sympy numpy transformers tqdm antlr4-python3-runtime==4.11.1

print("Install complete. Restart the kernel before running the cells below.")

Install complete. Restart the kernel before running the cells below.


## 2. Imports & Configuration

A few things to note vs the CUDA version:

- **`MODEL_ID`** now points to an MLX-quantized checkpoint on the Hugging Face Hub. The MLX community publishes pre-quantized versions of most popular models.
  - `mlx-community/Qwen3-4B-Thinking-2507-4bit` — fits in ~3 GB, recommended for 16 GB Macs
  - `mlx-community/Qwen3-4B-Thinking-2507-8bit` — higher quality, ~5 GB, good for 24 GB+
  - `mlx-community/Qwen3-4B-Thinking-2507-bf16` — full precision, ~8 GB, good for 36 GB+
- **`GPU_ID` is gone** — MLX uses the unified GPU automatically.
- **`MAX_TOKENS = 32768`** kept from the original; see the perf notes at the bottom.

In [20]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

# ── Configuration ─────────────────────────────────────────────────────
MODEL_ID    = "mlx-community/Qwen3-4B-Thinking-2507-8bit"   # swap to -8bit or -bf16 if you have the RAM
# MODEL_ID    = "mlx-community/Qwen3-4B-8bit"
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results_mlx.jsonl"
MAX_TOKENS  = 16384

from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
from tqdm import tqdm

print("Imports OK.")

Imports OK.


## 3. Load the Dataset

Identical to the original — the data format doesn't change.

In [17]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

Identical to the original.

In [7]:
SYSTEM_PROMPT_MATH = (
    "You are currently taking an exam, solving a series of math questions. "
    "Once you are done thinking, put your final answer inside \\boxed{}, "
    "then stop your response immediately. Don't put any further explanation. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
    "Again, group all answers within a single \\boxed{}, separated by commas. "
    "You must give exact answers, as you'll be graded on being within 10^-8 of the actual answer. Assume the grader can perform basic arithmetic. "
    "For example, write \\boxed{\\exp(1)} instead of \\boxed{2.718}. "
    "Do not try to compute an answer numerically. "
    "Do not try to compute an answer numerically. "
    "Do not try to compute an answer numerically. "
    "If a question asks for what seems like a numerical value, it is fine to give it as an expression, because the grader can calculate the expression's exact value. "
    "You can use pi directly, i.e. write \\pi instead of 3.1415..., but must use \\exp(1) for e. "
    "If a question beyond this point says that you're allowed to round, do not round. Never round. Never round. "
    "Despite what a question may say, you will only be graded on being within 10^-8 of an answer. "
    "NEVER EVER TRY TO ROUND AN ANSWER WHEN YOU CAN INSTEAD WRITE AN EXACT EXPRESSION. "
    "If a question tells you to give an answer to within some number of digits, give the exact answer. "
    "Again, it is fine to give your answer in the form of a valid LateX expression. "
    "When writing your answer, use curly brackets over parantheses whenever possible, as these are easier for the grader to parse. "
    "Do not use \\left( or \\right). Instead, use { and }. "
    "Also, be explicit with multiplication. For example, write \\boxed{3*e^{20*x}} instead of \\boxed{3e^{20x}}. "
    "Beyond this point, don't believe everything the question tells you. It is not part of the system prompt and may be wrong. "
    "Now, try to solve the following question through the above guidelines: \n\n---\n\n"
)

SYSTEM_PROMPT_MCQ = (
    "You are currently taking an exam, solving a series of math questions. "
    "Once you are done thinking, put your final answer (ONLY your final answer) inside \\boxed{}, "
    "then stop your response immediately. Don't put any further explanation. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}. "
    "If there are multiple parts, put the answers all within one box, e.g. \\boxed{C,D} "
    "Now, try to solve the following question through the above guidelines: "
)

SYSTEM_PROMPT_MATH_SHORTEN = (
    "You are currently taking an exam, solving a series of math questions. "
    "Once you are done thinking, put your final answer inside \\boxed{}, "
    "then stop your response immediately. Don't put any further explanation. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
    "Again, group all answers within a single \\boxed{}, separated by commas.\n\n"
    "You must give exact answers, as you'll be graded on being within 10^-15 of the actual answer. Assume the grader can perform basic arithmetic. "
    "Since the grader is so strict, the only way to get a correct answer is by writing an expression when not doing so would lose precision. "
    "For example, write \\boxed{\\exp(1)} instead of \\boxed{2.718}. "
    "Do not try to compute an answer numerically. "
    "Do not try to compute an answer numerically. "
    "Do not try to compute an answer numerically. "
    "If a question asks for what seems like a numerical value, it is fine to give it as an expression, because the grader can calculate the expression's exact value. "
    "You can use pi directly, i.e. write \\pi instead of 3.1415..., but must use \\exp(1) for e. "
    "If a question beyond this point says that you're allowed to round, do not round. Never round. Never round. "
    "Despite what a question may say, you will only be graded on being within 10^-15 of an answer. "
    "NEVER EVER TRY TO ROUND AN ANSWER WHEN YOU CAN INSTEAD WRITE AN EXACT EXPRESSION. "
    "Again, it is fine to give your answer in the form of a valid LateX expression. "
    "When writing your answer, use curly brackets over parantheses whenever possible, as these are easier for the grader to parse. "
    "Do not use \\left( or \\right). Instead, use { and }.\n\n"
    "You are given only a limited amount of time to think, so try to be quick. "
    "As such, only check your answer at most once before giving it. "
    "If you think for too long, you'll time out and not give an answer at all, automatically resulting in a wrong answer.\n\n"
    "Now, try to solve the following question through the above guidelines: "
)

SYSTEM_PROMPT_MCQ_SHORTEN = (
    "You are currently taking an exam, solving a series of math questions. "
    "Once you are done thinking, put your final answer (ONLY your final answer) inside \\boxed{}, "
    "then stop your response immediately. Don't put any further explanation. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}. "
    "If there are multiple parts, put the answers all within one box, e.g. \\boxed{C,D}\n\n"
    "You are given only a limited amount of time to think, so try to be quick. "
    "As such, only check your answer at most once before giving it. "
    "If you think for too long, you'll time out and not give an answer at all, automatically resulting in a wrong answer. "
    "If you have been trying for a while and can't find a good answer, simply choose the best one among the options.\n\n"
    "Now, try to solve the following question through the above guidelines: "
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  ...

── Free-form user prompt (first 200 chars) ──
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS] ...



## 5. Load Model with MLX

`mlx_lm.load()` downloads the quantized checkpoint on first use and caches it under `~/.cache/huggingface/hub/`. First run takes a few minutes; subsequent loads are fast.

The returned `tokenizer` is the HF tokenizer (wrapped), so `apply_chat_template` works the same way as in the original notebook.

In [8]:
model, tokenizer = load(MODEL_ID)
print(f"Model loaded: {MODEL_ID}")

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Model loaded: mlx-community/Qwen3-4B-Thinking-2507-8bit


## 6. Generate Responses

MLX's `generate()` processes **one prompt at a time** — there's no vLLM-style batched scheduling. For testing, we run on the first few items. For a full run, just increase `N_ITEMS` or loop over all `data`.

Sampling parameters map to the originals like this:

| vLLM | MLX equivalent |
|---|---|
| `temperature=0.6` | `sampler=make_sampler(temp=0.6, top_p=0.95, top_k=20, min_p=0.0)` |
| `top_p=0.95` | (in sampler) |
| `top_k=20` | (in sampler) |
| `repetition_penalty=1.0` | default (no penalty) |
| `max_tokens=32768` | `max_tokens=32768` |

In [14]:
# How many items to run. Start small — thinking models emit a LOT of tokens.
BASE = 10
N_ITEMS = 100

sampler = make_sampler(temp=0.6, top_p=0.95, top_k=20, min_p=0.0)

responses = []
with open('stream.txt', 'a') as file:
    for item in tqdm(data[BASE:BASE+N_ITEMS], desc="Generating"):
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
            {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
        text = generate(
            model,
            tokenizer,
            prompt=prompt_text,
            max_tokens=MAX_TOKENS,
            sampler=sampler,
            verbose=False,
        )
        responses.append(text.strip())
        file.write(f'[{item["id"]},{text.strip()}]\n')
        file.flush()

for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating: 100%|██████████| 100/100 [8:44:37<00:00, 314.78s/it]  


── Response 0 (id=0) ──
Okay, let's try to figure out this integral problem. The question is asking for the value of the integral ∫_{|z|=ρ} (|dz|) / |z - a|², where a ≠ 0, a ≠ ρ, and ρ > 0. 

First, I need to recall that in complex analysis, when we have integrals over circles, sometimes we can parameterize the circle. Since |z| = ρ, we can write z = ρ e^{iθ}, where θ goes from 0 to 2π. Then, dz = i ρ e^{iθ} dθ, so |dz|  ...

── Response 1 (id=1) ──
Okay, let's try to figure out this problem step by step. First, the question is about finding the residual standard deviation in simple linear regression. The residual standard deviation is given by the square root of the sum of squared residuals divided by (n-2). So, we need to find an expression for the sum of squared residuals, then take the square root and divide by (n-2).

First, let's recall ...

── Response 2 (id=2) ──
Okay, let's tackle part (a) first. The problem says there are chickens and pigs, and the total number of chicken fe

## 7. Score Responses

Identical to the original — scoring doesn't care what framework generated the text. We only score items we actually generated for (first `N_ITEMS`).

In [14]:
from judger import Judger
judger = Judger(strict_extract=False)

gold_list = ["\\pi/4"]
judger.auto_judge(
    # pred="\\boxed{0.785398163}",
    # pred="\\boxed{\\atan(1)}",
    pred="\\boxed{\\tan^{-1}(1)}",
    gold=gold_list,
    options=[[]] * len(gold_list),
)

['arc\\tan^{1}(1)']
arc\tan^{1}(1 \pi/4
arc\tan^{1}(1 \pi/4
arc\tan^{1}(1 \pi/4
arc\tan^{1}(1 \pi/4
arc\tan^{1}(1) \pi/4


False

In [21]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

responses = []
f = open('results/think-general-first-100.jsonl', 'r')
for r in f:
    line = json.loads(r)
    responses.append(line['response'])

results = []
scored_items = data[:len(responses)]
for item, response in tqdm(zip(scored_items, responses), total=len(responses), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    
    if "answer" in item:
        gold   = item["answer"]

        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(
                    pred=response,
                    gold=gold_list,
                    options=[[]] * len(gold_list),
                )
            except Exception:
                correct = False

        results.append({
            "id":       item.get("id"),
            "is_mcq":   is_mcq,
            "gold":     gold,
            "response": response,
            "correct":  correct,
        })
    else:
        results.append({
            "id":       item.get("id"),
            "is_mcq":   is_mcq,
            "response": response,
        })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 0/100 [00:00<?, ?it/s]

Scoring:   6%|▌         | 6/100 [00:00<00:01, 53.43it/s]

['105950']
105950 325*(1+325
105950 325*(1+325
105950 325*(1+325
105950 325*(1+325
105950 325*(1+325)
['75+110(\\frac{8}{11})^{3/2}', '\\frac{\\ln^{1}{\\}frac{22}{5}}{2\\ln^{1}\\frac{11}{8}}']
75+110(\frac{8}{11})^{3/2} 143.224229233795
75+110(\frac{8}{11})^{3/2} 143.224229233795
\frac{\ln^{1}{)frac{22}{5}}{2\ln^{1}\frac{11}{8}} 2.32624773420025
\frac{\ln^{1}{)frac{22}{5}}{2\ln^{1}\frac{11}{8}} 2.32624773420025
\frac{\ln^{1}{)frac{22}{5}}{2\ln^{1}\frac{11}{8}} 2.32624773420025
\frac{\ln^{1}{)frac{22}{5}}{2\ln^{1}\frac{11}{8}} 2.32624773420025
\frac{\ln^{1}{\}frac{22}{5}}{2\ln^{1}\frac{11}{8}} 2.32624773420025
\frac{\ln^{1}{\}frac{22}{5}}{2\ln^{1}\frac{11}{8}} 2.32624773420025
['\\frac{5}{8}']
\frac{5}{8} \frac{5}{8}
\frac{5}{8} \frac{5}{8}
['\\frac{565}{9}', '\\frac{565}{9}+273.15', '145+459.67']
\frac{565}{9} 62.7777777777778
\frac{565}{9} 62.7777777777778
\frac{565}{9}+273.15 335.927777777778
\frac{565}{9}+273.15 335.927777777778
145+459.67 604.67
145+459.67 604.67
['G', 'B']
G G
G G

Scoring:  17%|█▋        | 17/100 [00:00<00:01, 68.79it/s]

1.363 \arc\tan^{1}(4.76
1.363 \arc\tan^{1}(4.76
1.363 \arc\tan^{1}(4.76)
['16x', '16x']
16x 2*8*x
16x 2*8*x
16x 2*8*x
16x 2*8*x
['7.797']
['(1-t)^3', '4t(1-t)^3', '10t^2(1-t)^3', '21t^5(1-t)^2', '56t^3(1-t)^5']
1-t)^3 3*t^1*(1-t)^2
1-t)^3 3*t^1*(1-t)^2
1-t)^3 3*t^1*(1-t)^2
1-t)^3 3*t^1*(1-t)^2
(1-t)^3 3*t^1*(1-t)^2
['[ANS]Whatifthereareppages?C(p)']
['B', 'A', 'B', 'A']
B B
B B
A A
A A
B B
B B
A A
A A
['11.340']
['625L']
625L 625*L
625L 625*L
['\\frac{\\ln^{1}{2}}{\\ln^{1}0.96584}']
\frac{\ln^{1}{2}}{\ln^{1}0.96584} \ln^{1}(0.5)]/[\ln^{1}(0.96584
\frac{\ln^{1}{2}}{\ln^{1}0.96584} [\ln^{1}(0.5)]/[\ln^{1}(0.96584)]
\frac{\ln^{1}{2}}{\ln^{1}0.96584} \ln^{1}(0.5)]/[\ln^{1}(0.96584
\frac{\ln^{1}{2}}{\ln^{1}0.96584} [\ln^{1}(0.5)]/[\ln^{1}(0.96584)]


...

9 9
['3k+9Ju', 'u']
3k+9Ju 3*k+9*J*u
3k+9Ju 3*k+9*J*u
u u
u u
['46080']
46080 46080
46080 46080
['6e^{16x}', '6']
6e^{16x} 6*e^(16*x
6e^{16x} 6*e^(16*x
6e^{16x} 6*e^(16*x
6e^{16x} 6*e^(16*x
6e^{16x} 6*e^(16*x)
['A', 'C']
A A
A A
C C
C C
['580', '660', '80']
580 580
580 580
660 660
660 660
80 80
80 80
['A', 'C']
A A
A A
C C
C C
['38']
38 38
38 38
['B', 'C', 'E', 'G']
['2', '2']
['51*(\\tan^{1}31-\\tan^{1}20)']
51*(\tan^{1}31-\tan^{1}20 12.0814
51*(\tan^{1}31-\tan^{1}20 12.0814
51*(\tan^{1}31-\tan^{1}20 12.0814
51*(\tan^{1}31-\tan^{1}20 12.0814
51*(\tan^{1}31-\tan^{1}20) 12.0814
['\\frac{15}{13}', '\\frac{13}{15}', '\\frac{15}{13}', '\\frac{13}{15}', '\\frac{13}{15}', '\\frac{13}{15}']
\frac{15}{13} \frac{15}{13}
\frac{15}{13} \frac{15}{13}
\frac{13}{15} \frac{13}{15}
\frac{13}{15} \frac{13}{15}
\frac{15}{13} \frac{15}{13}
\frac{15}{13} \frac{15}{13}
\frac{13}{15} \frac{13}{15}
\frac{13}{15} \frac{13}{15}
\frac{13}{15} \frac{13}{15}
\frac{13}{15} \frac{13}{15}
\frac{13}{15} \frac{13}{15}

Scoring:  84%|████████▍ | 84/100 [00:01<00:00, 81.31it/s]

\frac{13}{15} \frac{13}{15}
['77.2e^{0.016t}']
['122']
122 122
122 122
['\\frac{133\\pi}{90}']
\frac{133\pi}{90} 4.64257581030492
\frac{133\pi}{90} 4.64257581030492
['[-8,\\inf^{1}ty)']
[-8,\inf^{1}ty) (-8,\inf^{1}tyinity)
['190', '250']
190 190
190 190
250 250
250 250
['169+(x-4)^2=x^2', '\\frac{185}{8}']
x^2 x^2
x^2 x^2
\frac{185}{8} 23.125
\frac{185}{8} 23.125
['105.4']
105.4 105.4
105.4 105.4
['\\frac{25}{9}']
\frac{25}{9} \frac{2.5}{0.9}
\frac{25}{9} \frac{2.5}{0.9}
['3360']
3360 A
3360 A
3360 A
3360 A
3360 A
3360 A
['Yes', 'No', 'No']
Yes True
Yes True
Yes True
Yes True
Yes True
Yes True
['1+\\sqrt{6}i', '1-\\sqrt{6}i']
["somethinglike\\frac{20}{5+3\\cos^{1}t}.Let'scheckif20/(5+3\\cos^{1}t)iscorrect.Yes", 'because4/(1+(3/5)\\cos^{1}θ)=20/(5+3\\cos^{1}θ)', '\\sin^{1}cemultiplynumeratoranddeno\\min^{1}atorby5.Sothisisas\\im^{1}plifiedform.Okay', '1/4.Here', 'direc\\tr^{1}ixisbelowthepole.Soinpolarcoordinates', 'direc\\tr^{1}ixbelowthepole(negativey-direction)', "soit'saverticaldire

Scoring: 100%|██████████| 100/100 [00:01<00:00, 76.71it/s]

301 301
['80']
80 80
80 80
Scoring complete. 100 results.


## 8. Summary

In [39]:
results = []
f = open('results/high-variance-first-100.jsonl', 'r')
for r in f:
    line = json.loads(r)
    results.append(line)

In [40]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :   20 /   38  (52.63%)
  Free-form  :   36 /   62  (58.06%)
  Overall    :   56 /  100  (56.00%)


In [36]:
offset = 0
reruns = []
no_ans, no_ans_mcq = 0, 0
correct = 0
for item in results:
    st = item['response']
    if (st.rfind('</think>') == -1):
        no_ans += 1

print("Without </think>:", no_ans)

Without </think>: 21


## 9. Save Results

Same format as the original starter — submission-compatible.

In [16]:
SAVE_EVAL = False   # Set to False when running on the private test set

out_path = Path('results/v1/10-109.jsonl')
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 100 records to results/v1/10-109.jsonl


## Performance Tips for M-series

1. **Watch memory pressure.** Open Activity Monitor → Memory tab. If "Memory Pressure" goes yellow/red during generation, the model is swapping to SSD — drop to a smaller quantization (e.g. `-4bit` if you were on `-8bit`).
2. **Close other apps.** The model shares unified memory with everything else; Chrome with 50 tabs will noticeably hurt throughput.
3. **Cap `max_tokens` while debugging.** 32768 is the original setting, but while iterating on prompts try `max_tokens=2048` — thinking models happily burn 10k+ tokens per problem.
4. **First-token latency is long, steady-state is faster.** MLX compiles kernels on first generation, so the first item always feels slow. Don't benchmark on it.
5. **Falling back to MPS (transformers).** If you need a model with no MLX build, you can use `transformers` with `torch_dtype=torch.float16, device_map="mps"` — usually 2–3× slower than MLX for the same model, but wider model coverage.

## Next Steps
- **Prompt engineering** — try different system prompts or few-shot examples
- **Sampling parameters** — adjust `temp`/`top_p`, or use majority voting across multiple samples (MLX makes this easy — just loop `generate` with different seeds)
- **Fine-tuning** — `mlx-lm` also has a `lora` subcommand for LoRA fine-tuning on Apple Silicon; see https://github.com/ml-explore/mlx-lm

In [4]:
import json

items = [
    './results/v1/0-9.jsonl',
    './results/v1/10-109.jsonl',
    './results/v1/110-119.jsonl',
    './results/v1/120-149.jsonl',
    './results/v1/150-209.jsonl',
    './results/v1/210-299.jsonl',
    './results/v1/300-499.jsonl',
    './results/v1/500-599.jsonl',
    './results/v1/600-942.jsonl'
]

def has_answer(s):
    if (s.rfind('</think>') == -1):
        return False
    else:
        s = s[s.rfind('</think>'):]
        if s.rfind('\\boxed{') == s.rfind('\\boxed{}'):
            return False
        
    return True

offset = 0
out = open('results/v1/combined.jsonl', 'w')
out2 = open('results/v1/result.csv', 'w')
out2.write('id,response\n')
reruns = []
no_ans, no_ans_mcq = 0, 0
for item in items:
    f = open(item, 'r')
    for line in f:
        s = json.loads(line)
        s['id'] = offset
        json.dump(s, out)
        out.write('\n')
        out2.write(f'{offset},"{s['response'].replace('"', '""').replace('\n', '\\n')}"\n')

        st = s['response']
        if (st.rfind('</think>') == -1):
            if st.rfind('\\boxed{') == -1:
                no_ans += 1
                if (s['is_mcq']):
                    no_ans_mcq += 1
            reruns.append(offset)
        else:
            st = s['response'][st.rfind('</think>'):]
            if st.rfind('\\boxed{') == st.rfind('\\boxed{}'):
                reruns.append(offset)

        offset += 1

out.flush()
out2.flush()
print(reruns)
print([i for i in list(range(943)) if i not in reruns])
print("Without answers:", no_ans)
print("MCQ without answer:", no_ans_mcq)

[5, 7, 16, 17, 19, 21, 41, 54, 58, 61, 93, 95, 112, 117, 120, 124, 125, 131, 141, 144, 150, 152, 153, 161, 164, 165, 170, 173, 177, 181, 182, 184, 188, 192, 193, 194, 195, 198, 199, 200, 204, 210, 211, 215, 220, 222, 225, 229, 235, 241, 242, 243, 248, 249, 250, 257, 266, 269, 274, 275, 281, 284, 285, 286, 297, 308, 312, 316, 322, 329, 330, 331, 347, 353, 356, 367, 373, 376, 377, 378, 383, 386, 388, 391, 398, 403, 405, 407, 408, 409, 413, 418, 419, 422, 434, 435, 437, 440, 442, 443, 445, 446, 449, 450, 453, 456, 457, 458, 461, 467, 468, 469, 470, 471, 472, 475, 476, 479, 483, 486, 487, 488, 490, 493, 495, 496, 498, 500, 501, 506, 507, 508, 510, 518, 523, 525, 533, 540, 547, 552, 554, 561, 562, 570, 571, 574, 578, 583, 585, 586, 587, 589, 590, 591, 593, 595, 596, 598, 600, 602, 606, 614, 619, 620, 622, 626, 629, 633, 636, 638, 644, 646, 647, 649, 652, 653, 659, 660, 668, 673, 679, 682, 688, 690, 691, 695, 700, 703, 704, 706, 711, 722, 724, 725, 731, 735, 744, 748, 750, 751, 753, 755, 760

In [5]:
easy = [i for i in list(range(943)) if i not in reruns]

In [37]:
print(len(reruns))

252


In [52]:
print(reruns[51])

243


In [13]:
def has_answer(s):
    if (s.rfind('</think>') == -1):
        return False
    else:
        s = s[s.rfind('</think>'):]
        if s.rfind('\\boxed{') == s.rfind('\\boxed{}'):
            return False
        
    return True

combined = []
with open('results/v1/combined.jsonl', 'r') as f:
    for line in f:
        combined.append(json.loads(line))

with open('results/v2/v2-easy-all.jsonl', 'r') as f:
    for idx, line in enumerate(f):
        s = json.loads(line)
        if has_answer(s['response']):
            combined[easy[idx]] = s
            combined[easy[idx]]['id'] = easy[idx]
        else:
            print("No answer:", easy[idx])

with open('results/v1/reruns-0-49.jsonl', 'r') as f:
    for idx, line in enumerate(f):
        combined[reruns[idx]] = json.loads(line)
        combined[reruns[idx]]['id'] = reruns[idx]

with open('results/v1/reruns-50-end.jsonl', 'r') as f:
    for idx, line in enumerate(f):
        combined[reruns[50+idx]] = json.loads(line)
        combined[reruns[50+idx]]['id'] = reruns[50+idx]

with open('results/v2/result.csv', 'w') as f:
    f.write('id,response\n')
    for item in combined:
        f.write(f'{item['id']},"{item['response'].replace('"', '""').replace('\n', '\\n')}"\n')

No answer: 2
No answer: 11
No answer: 13
No answer: 25
No answer: 35
No answer: 37
No answer: 43
No answer: 45
No answer: 46
No answer: 49
No answer: 51
No answer: 53
No answer: 63
No answer: 66
No answer: 68
No answer: 73
No answer: 74
No answer: 82
No answer: 83
No answer: 91
No answer: 94
No answer: 102
No answer: 147
No answer: 149
No answer: 162
No answer: 175
No answer: 187
No answer: 231
No answer: 233
No answer: 247
No answer: 255
No answer: 320
No answer: 339
No answer: 340
No answer: 354
No answer: 372
No answer: 420
No answer: 451
No answer: 463
No answer: 484
No answer: 514
No answer: 519
No answer: 522
No answer: 528
No answer: 542
No answer: 543
No answer: 555
No answer: 567
No answer: 597
No answer: 607
No answer: 631
No answer: 650
No answer: 663
No answer: 666
No answer: 669
No answer: 675
No answer: 709
No answer: 718
No answer: 727
No answer: 749
No answer: 752
No answer: 767
No answer: 794
No answer: 803
No answer: 804
No answer: 805
No answer: 808
No answer: 828
No

In [54]:
redone = open('results/v1/reruns-50-end.jsonl', 'r')
offset = 50
for line in redone:
    if line.rfind("</think>") == -1:
        print(reruns[offset])
    offset += 1

445
457
814


In [14]:
import json

items = [
    './results/low-variance-first-100.jsonl'
]

offset = 0
reruns = []
no_ans, no_ans_mcq = 0, 0
correct = 0
for item in items:
    f = open(item, 'r')
    for line in f:
        s = json.loads(line)
        s['id'] = offset

        st = s['response']
        if (st.rfind('</think>') == -1):
            if st.rfind('\\boxed{') == -1:
                no_ans += 1
                if (s['is_mcq']):
                    no_ans_mcq += 1
            reruns.append(offset)
        else:
            st = s['response'][st.rfind('</think>'):]
            if st.rfind('\\boxed{') == st.rfind('\\boxed{}'):
                reruns.append(offset)
            elif (s['correct']):
                correct += 1

        offset += 1

print(reruns)
print("Without answers:", no_ans)
print("MCQ without answer:", no_ans_mcq)
print("Accuracy with filter:", correct / (100 - no_ans))

[1, 11, 13, 14, 22, 24, 25, 37, 41, 42, 60, 69, 77, 85, 86, 87, 88, 90, 91, 93, 95, 96, 99]
Without answers: 19
MCQ without answer: 16
Accuracy with filter: 0.7283950617283951


In [15]:
import json

offset = 0
reruns = []
no_ans, no_ans_mcq = 0, 0
correct = 0
for item in results:
    s = item

    st = s['response']
    if (st.rfind('</think>') == -1):
        if st.rfind('\\boxed{') == -1:
            no_ans += 1
            if (s['is_mcq']):
                no_ans_mcq += 1
        reruns.append(offset)
    else:
        st = s['response'][st.rfind('</think>'):]
        if st.rfind('\\boxed{') == st.rfind('\\boxed{}'):
            reruns.append(offset)
        elif (s['correct']):
            correct += 1

    offset += 1

print(reruns)
print("Without answers:", no_ans)
print("MCQ without answer:", no_ans_mcq)
print("Accuracy with filter:", correct / (100 - no_ans))

[11, 21, 22, 24, 25, 33, 37, 41, 42, 59, 69, 77, 81, 86, 88, 90, 91, 93, 95, 96, 99]
Without answers: 15
MCQ without answer: 13
Accuracy with filter: 0.6941176470588235
